In [2]:
import torch
import torch.nn as nn

In [3]:
class BirdCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
          nn.Conv2d(3, 32, kernel_size=3, padding=1),
          nn.BatchNorm2d(32),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.block2 = nn.Sequential(
                  nn.Conv2d(32, 64, kernel_size=3, padding=1),
                  nn.BatchNorm2d(64),
                  nn.ReLU(),
                  nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.block3 = nn.Sequential(
                          nn.Conv2d(64, 128, kernel_size=3, padding=1),
                          nn.BatchNorm2d(128),
                          nn.ReLU(),
                          nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 200)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x


In [4]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):

    model.train()

    total_loss = 0
    total_correct = 0
    total_images = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()

        total_images += images.size(0)

    avg_loss = total_loss / total_images
    accuracy = 100 * total_correct / total_images

    return avg_loss, accuracy

In [5]:
def validate(model, val_loader, criterion, device):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_images = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            total_correct += (predictions == labels).sum().item()

            total_images += images.size(0)

    avg_loss = total_loss / total_images
    accuracy = 100 * total_correct / total_images

    return avg_loss, accuracy

In [7]:
from torch.utils.data import TensorDataset, DataLoader

images = torch.randn(64, 3, 224, 224)
labels = torch.randint(0, 200, (64,))

dataset = TensorDataset(images, labels)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(dataset, batch_size=16, shuffle=False)

In [ ]:

import matplotlib.pyplot as plt

# Select CPU or GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create model
model = BirdCNN().to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
best_val_acc = -1.0

# Store training history
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):

    # Training
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validation
    val_loss, val_acc = validate(
        model, val_loader, criterion, device
    )

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    # Print results
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_bird_cnn.pth")
        print("Best model saved!")

# Plot training and validation loss
epochs = range(1, num_epochs + 1)

plt.figure()
plt.plot(epochs, train_losses, label="Training Loss")
plt.plot(epochs, val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# Plot training and validation accuracy
plt.figure()
plt.plot(epochs, train_accuracies, label="Training Accuracy")
plt.plot(epochs, val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
